In [ ]:
from openai import OpenAI
import json
import requests

In [ ]:
model="gpt-4.1-mini"

In [ ]:
client = OpenAI(
    api_key="api"
)

In [ ]:
def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    data = response.json()
    return data['current']['temperature_2m']


client = OpenAI(
    api_key="YOUR_API_KEY_HERE"
)

weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": { "type": "number", "description": "Latitude of the location." },
            "longitude": { "type": "number", "description": "Longitude of the location." }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}


tools = [weather_tool]

In [ ]:
messages = [{"role": "developer", "content": "What's the weather like in arak today?"}]

while True:
    response = client.responses.create(
        model=model,
        input=messages,
        tools=tools
    )

    if response.output:
        for output_item in response.output:
            if hasattr(output_item, 'type') and output_item.type == "function_call":
                messages.append(output_item)

                tool_call = output_item
                args = json.loads(tool_call.arguments)

                result = get_weather(args['latitude'], args['longitude'])
                print(f"Executed {tool_call.name}: Result = {result}°C")

                messages.append({
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": str(result)
                })

    if hasattr(response, 'output_text') and response.output_text:
        print("Final Agent Output:", response.output_text)
        break


    if not response.output:
        break

In [ ]:
def agent_loop(messages, tools):
    while True:
        response = client.responses.create(
            model=model,
            input=messages,
            tools=tools
        )

        if response.output:
            for output_item in response.output:
                if hasattr(output_item, 'type') and output_item.type == "function_call":
                    messages.append(output_item)

                    tool_call = output_item
                    args = json.loads(tool_call.arguments)

                    result = get_weather(args['latitude'], args['longitude'])
                    print(f"Executed {tool_call.name}: Result = {result}°C")

                    messages.append({
                        "type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": str(result)
                    })

        if hasattr(response, 'output_text') and response.output_text:
            print("Final Agent Output:", response.output_text)
            break


        if not response.output:
            break

In [ ]:
messages = [{"role": "developer", "content": "What's the weather like in arak today? Before replying I want you to also get the weather for Berlin."}]

In [ ]:
import json

def objective_met(search_count, max_searches=5):
    return search_count > max_searches

messages= [
    {
        "role": "developer",
        "content": (
            "Your goal is to gather weather for at least 5 different cities. "
            "Once you've done that, respond with 'task complete'."
        )
    },
    {"role": "user", "content": "Search the weather in yazd."}
]

search_count = 0

while True:
    response = client.responses.create(
        model=model,
        input=messages,
        tools=tools
    )

    for item in response.output or []:
        if getattr(item, 'type', None) == "function_call":
            messages.append(item)

            args = json.loads(item.arguments)
            temp = get_weather(args['latitude'], args['longitude'])
            print(f"Executed {item.name}: {temp}°C")

            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": str(temp)
            })

            search_count += 1

    if hasattr(response, 'output_text') and response.output_text:
        text = response.output_text
        print("Agent says:", text)
        messages.append({"role": "assistant", "content": text})

    if objective_met(search_count):
        print(f"Objective met: searched {search_count} cities. Task complete.")
        break

    messages.append({
        "role": "user",
        "content": f"We've searched {search_count} so far. Please search another city."
    })

In [ ]:
print(messages[-1])